### Sentencia "Merge"
> - Utiliza la operación "Upsert" (Insert, Update y Delete) en una sola instrucción.
> - Permite fusionar nuevos datos en una tabla de destino basándose en una condición de coincidencia.

Documentación - [MERGE](https://learn.microsoft.com/es-es/azure/databricks/sql/language-manual/delta-merge-into)

In [0]:
SELECT * FROM demo.delta_lake.raw_products_inventory;

#### Crear la tabla para fusionar los datos

In [0]:
CREATE TABLE IF NOT EXISTS demo.delta_lake.products_inventory
(
  product_id STRING,
  product_name STRING,
  production_date DATE,
  expiration_date DATE,
  quantity INT,
  category STRING
);

#### Combinar los datos de origen en la tabla de destino.
1. Insertar nuevos productos recibidos
2. Actualizar el "name", "production date", "expiration date", "quantity" y "category" si se reciben actualizaciones.
3. Eliminar productos que han "expirado" (status = 'Expired')

In [0]:
MERGE INTO demo.delta_lake.products_inventory AS tgt
USING demo.delta_lake.raw_products_inventory AS src
ON tgt.product_id = src.product_id
WHEN MATCHED AND src.status = 'Available' THEN
  UPDATE SET tgt.product_name = src.product_name,
             tgt.production_date = src.production_date,
             tgt.expiration_date = src.expiration_date,
             tgt.quantity = src.quantity,
             tgt.category = src.category
WHEN MATCHED AND src.status = 'Expired' THEN
  DELETE
WHEN NOT MATCHED AND src.status = 'Available' THEN
  INSERT (product_id, product_name, production_date, expiration_date, quantity, category)
  VALUES (src.product_id, src.product_name, src.production_date, src.expiration_date, src.quantity, src.category);

In [0]:
SELECT * FROM demo.delta_lake.products_inventory;